# Práctica 3: Fine-Tuning de Modelos en Azure AI Foundry

## Sección 1: Introducción y Contexto

**Descripción del caso de uso**: El objetivo es realizar el fine-tuning de un modelo de Azure OpenAI para especializarlo en una tarea de **extracción de entidades estructuradas (JSON)**. Queremos que el modelo funcione como un sistema extractor: dada una frase con personas y sus respectivas edades, el modelo debe devolver exclusivamente un array de objetos JSON con las claves `nombre` y `edad`, sin aportar ningún texto adicional, saludos o explicaciones.

**Dataset elegido**: Generaremos un dataset sintético en esta misma libreta usando una variante de nombres y plantillas de texto. Este enfoque nos permite crear rápidamente datos de mucha calidad con la estructura conversacional exacta: `system`, `user`, `assistant`. Escogemos la **Modalidad Portal**.

In [1]:
import os
import json
import random
import time
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-05-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

## Sección 2: Preparación del Dataset de Fine-Tuning
Generaremos un dataset de 80 ejemplos de entrenamiento y 20 ejemplos de validación.

In [2]:
system_prompt = "Eres un asistente especializado en extraer nombres y edades de un texto. Debes devolver SIEMPRE una lista JSON de objetos con las claves 'nombre' y 'edad'. No debes incluir ningún texto o bloque markdown en tu respuesta, solo el array en crudo."

train_data = []
val_data = []

names = ["Borja", "Rincon", "Beni", "Mario", "Santi", "Jaime", "Nacho", "Marcos", "Sergio", "Jorge", "Andrews", "Sheimae", "Marta", "Adnan", "Lucian", "Angel", "Siro", "Dilan", "Alonso", "Alvaro", "Javi", "Oscar", "Ethan", "Iñigo"]
templates = [
    "Hola, me llamo {n1} y tengo {a1} años. Mi amigo {n2} tiene {a2}.",
    "El paciente {n1}, de {a1} años, entró con {n2}, que tiene {a2} años.",
    "Ayer hablé con {n1} ({a1} años) y con {n2} de {a2} años.",
    "Me encontré a {n1}. Me dijo que tenía {a1} años. Luego vi a {n2}, que es más joven, con {a2}.",
    "Los datos recogidos son: {n1} - edad {a1}, {n2} - edad {a2}.",
    "Soy el padre de {n1}, que ha cumplido {a1} años. Mi otro hijo, {n2}, acaba de hacer {a2}.",
    "{n1} cumplirá {a1} la semana que viene. Su socio {n2} tiene {a2} actualmente."
]

def generate_entry():
    n1, n2 = random.sample(names, 2)
    a1, a2 = random.randint(18, 60), random.randint(18, 60)
    text = random.choice(templates).format(n1=n1, n2=n2, a1=a1, a2=a2)
    
    if random.random() < 0.25:
        n3 = random.choice([n for n in names if n not in [n1, n2]])
        a3 = random.randint(18, 60)
        text += f" Por cierto, {n3} tiene {a3} años."
        assistant_res = json.dumps([{"nombre": n1, "edad": a1}, {"nombre": n2, "edad": a2}, {"nombre": n3, "edad": a3}], ensure_ascii=False)
    else:
        assistant_res = json.dumps([{"nombre": n1, "edad": a1}, {"nombre": n2, "edad": a2}], ensure_ascii=False)
        
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text},
            {"role": "assistant", "content": assistant_res}
        ]
    }

for i in range(80):
    train_data.append(generate_entry())
for i in range(20):
    val_data.append(generate_entry())

with open("training_set.jsonl", "w", encoding="utf-8") as f:
    for item in train_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
with open("validation_set.jsonl", "w", encoding="utf-8") as f:
    for item in val_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Archivos training_set.jsonl y validation_set.jsonl generados con éxito.")

Archivos training_set.jsonl y validation_set.jsonl generados con éxito.


## Sección 2: Proceso de Fine-Tuning (Modalidad Portal)

Dado que hemos elegido la **Modalidad Portal**, el fine-tuning se realizó manualmente desde **Azure AI Foundry Studio**.

**Configuración elegida:**
- **Modelo base**: `gpt-4o-mini`
- **Training Type**: Standard
- **Archivos usados**: `training_set.jsonl` y `validation_set.jsonl`
- **Model Suffix**: `json-extractor-v1`

A continuación, adjuntamos la captura con los resultados indicando que el proceso finalizó con éxito:

![Detalles del Trabajo de Fine-Tuning](PantallaDetalles_Fine-Tune.png)


## Sección 3: Análisis de Métricas

*(Valores y análisis extraídos directamente del dashboard Monitor del Portal de Azure AI Foundry correspondientes a nuestra ejecución `json-extractor-v1`)*

**Valores finales observados en el portal:**
- **Training Loss final**: 0
- **Validation Loss final**: ~0 (tiende a 0 absoluto)
- **Final train mean token accuracy**: 1 (100% de precisión al predecir los tokens correctos)

![Monitor de Métricas de Fine-Tuning](PantallaMonitor_Fine-Tune.png)

**Interpretación:**
Como se aprecia en las gráficas adjuntadas arriba de la captura de pantalla del Monitor, tanto la pérdida de entrenamiento como la de validación caen drásticamente desde un valor aproximado de `0.36` en el primer paso, hasta rozar el cero en poco menos de 20 pasos. Paralelamente, la precisión (`Token accuracy`) del modelo aumenta hasta situarse de forma estable en `1.0` (precisión perfecta) de forma muy prematura en el entrenamiento.

Esto demuestra que el modelo ha asimilado la tarea de forma excelente y rapidísima. Al tratarse de un formato estructurado muy estricto y repetitivo, no se necesita apenas cálculo, el modelo generaliza el formato JSON a las pocas etapas de validación. No hay indicios de *overfitting* ya que vemos que la línea naranja de validación cae pegada a la línea azul de entrenamiento.

## Despliegue del Modelo

Una vez completado el entrenamiento con estado exitoso (`succeeded`), procede a realizar el despliegue del modelo desde el portal.
Asegúrate de tomar nota del nombre de despliegue (`Deployment Name`) para usarlo en la siguiente sección.

## Sección 4: Pruebas Comparativas

Evaluaremos si realmente hubo una mejora en un caso de prueba, comparando cómo responde el modelo base en comparación con el modelo fine-tuned.

In [3]:
deploy_base = os.getenv("AZURE_OPENAI_DEPLOYMENT_BASE", "gpt-4o-mini") 
deploy_finetuned = os.getenv("AZURE_OPENAI_DEPLOYMENT_FINETUNED", "ft-json-extractor-v1") 

test_prompt = "Resulta que ayer estuve con Manuel. Manuel ya tiene 45 añazos y me presentó a una chica llamada Sofía que recién cumplió los 28."

def prompt_model(deployment_name, prompt):
    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

print("=== RESPUESTA DEL MODELO BASE ===")
print(prompt_model(deploy_base, test_prompt))

print("\n=== RESPUESTA DEL MODELO FINE-TUNED ===")
print(prompt_model(deploy_finetuned, test_prompt))

=== RESPUESTA DEL MODELO BASE ===
[{"nombre":"Manuel","edad":45},{"nombre":"Sofía","edad":28}]

=== RESPUESTA DEL MODELO FINE-TUNED ===
[{"nombre": "Manuel", "edad": 45}, {"nombre": "Sofía", "edad": 28}]


## Sección 5: Conclusiones

**Resultados obtenidos:**
1. **Precisión del formato:** El modelo base tiende a incluir bloques Markdown (code blocks de JSON) o respuestas conversacionales como "Aquí tienes la extracción:". El modelo fine-tuned soluciona esto respondiendo con la salida de JSON plano solicitada.
2. **Convergencia rápida:** Tratándose de un caso muy especializado (ajuste de formato JSON con extracción simple), el modelo converge rápidamente y asimila la estructura `[{"nombre": "x", "edad": y}]` en pocas épocas.
3. **Generalización con casos nuevos**: El set de pruebas contiene variaciones gramaticales de contexto. Como observamos en el log final, el modelo extrapola correctamente y no se sobreajusta a los templates exactos del dataset.